In [1]:
import argparse
import copy
import os
import os.path as osp
import time
import warnings
from logging import log

import mmcv
import torch
from mmcv import Config, DictAction
from mmcv.runner import get_dist_info, init_dist
from mmcv.utils import get_git_hash
from mmdet import __version__
from mmdet.models import build_detector
from mmdet.utils import collect_env

from ssod.apis import get_root_logger, set_random_seed, train_detector
from ssod.datasets import build_dataset
from ssod.utils import patch_config

In [2]:
args = argparse.Namespace(verbose=False, verbose_1=False)

args.config = "/rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_finetune_batch2.py"
args.work_dir = "/rsrch5/home/trans_mol_path/cercan/data/BE/cell_detect/acformer/finetune/output_batch2"
args.seed = 0 

In [3]:
cfg = Config.fromfile(args.config)

In [4]:
cfg.data.train


{'type': 'CellDetDataset_Lizard_6class',
 'ann_file': '/rsrch5/home/trans_mol_path/cercan/data/BE/cell_detect/acformer/finetune/qpproj_batch2/finetuning/training/finetune_annot.json',
 'img_prefix': '/rsrch5/home/trans_mol_path/cercan/data/BE/cell_detect/acformer/finetune/qpproj_batch2/finetuning/training/roi',
 'pipeline': [{'type': 'LoadImageFromFile'},
  {'type': 'LoadAnnotations', 'with_bbox': True},
  {'type': 'RandomFlip', 'flip_ratio': 0.5},
  {'type': 'AddOriImg_Resize'},
  {'type': 'AutoAugment',
   'policies': [[{'type': 'Resize',
      'img_scale': [(800, 800)],
      'multiscale_mode': 'value',
      'keep_ratio': True}],
    [{'type': 'Resize',
      'img_scale': [(800, 800)],
      'multiscale_mode': 'value',
      'keep_ratio': True},
     {'type': 'RandomCrop',
      'crop_type': 'absolute_range',
      'crop_size': (800, 800),
      'allow_negative_crop': True},
     {'type': 'Resize',
      'img_scale': [(800, 800)],
      'multiscale_mode': 'value',
      'override':

In [5]:

# import modules from string list.
if cfg.get("custom_imports", None):
    from mmcv.utils import import_modules_from_strings

    import_modules_from_strings(**cfg["custom_imports"])
# set cudnn_benchmark
if cfg.get("cudnn_benchmark", False):
    torch.backends.cudnn.benchmark = True

# work_dir is determined in this priority: CLI > segment in file > filename
if args.work_dir is not None:
    # update configs according to CLI args if args.work_dir is not None
    cfg.work_dir = args.work_dir
elif cfg.get("work_dir", None) is None:
    # use config filename as default work_dir if cfg.work_dir is None
    cfg.work_dir = osp.join(
        "./work_dirs", osp.splitext(osp.basename(args.config))[0]
    )

In [6]:
cfg = patch_config(cfg)

In [7]:
# if args.launcher == "none":
distributed = True

In [8]:
mmcv.mkdir_or_exist(osp.abspath(cfg.work_dir))

In [9]:
cfg.dump(osp.join(cfg.work_dir, osp.basename(args.config)))

In [10]:
args.config

'/rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_finetune_batch2.py'

In [11]:

timestamp = time.strftime("%Y%m%d_%H%M%S", time.localtime())
log_file = osp.join(cfg.work_dir, f"{timestamp}.log")
logger = get_root_logger(log_file=log_file, log_level=cfg.log_level)

In [12]:

meta = dict()
# log env info
env_info_dict = collect_env()
env_info = "\n".join([(f"{k}: {v}") for k, v in env_info_dict.items()])
dash_line = "-" * 60 + "\n"
logger.info(logger.handlers)
logger.info("Environment info:\n" + dash_line + env_info + "\n" + dash_line)
meta["env_info"] = env_info
meta["config"] = cfg.pretty_text
# log some basic info
logger.info(f"Distributed training: {distributed}")
logger.info(f"Config:\n{cfg.pretty_text}")

2024-09-20 21:43:09,912 - mmdet.ssod - INFO - [<StreamHandler stderr (INFO)>, <FileHandler /rsrch5/home/trans_mol_path/cercan/data/BE/cell_detect/acformer/finetune/output_batch2/20240920_214308.log (INFO)>]
2024-09-20 21:43:09,914 - mmdet.ssod - INFO - Environment info:
------------------------------------------------------------
sys.platform: linux
Python: 3.8.10 (default, Jul 29 2024, 17:02:10) [GCC 9.4.0]
CUDA available: True
GPU 0,1: NVIDIA A100-SXM4-40GB
CUDA_HOME: /usr/local/cuda
NVCC: Cuda compilation tools, release 11.1, V11.1.105
GCC: x86_64-linux-gnu-gcc (Ubuntu 9.3.0-17ubuntu1~20.04) 9.3.0
PyTorch: 1.10.1+cu111
PyTorch compiling details: PyTorch built with:
  - GCC 7.3
  - C++ Version: 201402
  - Intel(R) Math Kernel Library Version 2020.0.0 Product Build 20191122 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.2.3 (Git Hash 7336ca9f055cf1bfa13efb658fe15dc9b41f0740)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NN

In [13]:

if args.seed is not None:
    logger.info(
        f"Set random seed to {args.seed}, " f"deterministic: True"
    )
    set_random_seed(args.seed, deterministic=True)
cfg.seed = args.seed
meta["seed"] = args.seed
meta["exp_name"] = osp.basename(args.config)

2024-09-20 21:43:15,871 - mmdet.ssod - INFO - Set random seed to 0, deterministic: True


In [14]:
model = build_detector(
        cfg.model, train_cfg=cfg.get("train_cfg"), test_cfg=cfg.get("test_cfg")
    )

/App/ACFormer/ssod/models/global_local_stn_sequence_plus.py:71: UserWarning: nn.init.uniform is now deprecated in favor of nn.init.uniform_.
  torch.nn.init.uniform(m.weight, a=-0.1, b=0.1)


In [29]:
model.init_weights()

2024-09-17 21:14:01,651 - mmdet.ssod - INFO - initialize ConvNeXt with init_cfg {'type': 'Pretrained', 'checkpoint': '/rsrch5/home/trans_mol_path/cercan/data/BE/cell_detect/acformer/finetune/output/epoch_21.pth', 'prefix': None}
2024-09-17 21:14:01,653 - mmcv - INFO - load model from: /rsrch5/home/trans_mol_path/cercan/data/BE/cell_detect/acformer/finetune/output/epoch_21.pth
2024-09-17 21:14:01,655 - mmcv - INFO - load checkpoint from local path: /rsrch5/home/trans_mol_path/cercan/data/BE/cell_detect/acformer/finetune/output/epoch_21.pth
2024-09-17 21:14:13,202 - mmcv - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: absolute_pos_embed, affine_token, globals.backbone.downsample_layers.0.0.weight, globals.backbone.downsample_layers.0.0.bias, globals.backbone.downsample_layers.0.1.weight, globals.backbone.downsample_layers.0.1.bias, globals.backbone.downsample_layers.1.0.weight, globals.backbone.downsample_layers.1.0.bias, globals.bac

In [31]:
datasets = [build_dataset(cfg.data.train), build_dataset(cfg.data.val)]


loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
loading annotations into memory...
Done (t=0.29s)
creating index...
index created!


In [32]:
cfg.checkpoint_config.meta = dict(
            mmdet_version=__version__ + get_git_hash()[:7], CLASSES=datasets[0].CLASSES
        )
cfg.checkpoint_config

{'by_epoch': True,
 'interval': 1,
 'meta': {'mmdet_version': '2.25.1cb866bf',
  'CLASSES': ('Neutrophil',
   'Epithelial',
   'Lymphocyte',
   'Plasma',
   'Eosinophil',
   'Connective')}}

In [33]:
model.CLASSES = datasets[0].CLASSES

In [34]:
train_detector(
        model,
        datasets,
        cfg,
        distributed=distributed,
        validate=True,
        timestamp=timestamp,
        meta=meta,
    )

RuntimeError: Default process group has not been initialized, please make sure to call init_process_group.

In [71]:
mmcv.device.get_device()


'cuda'

In [ ]:
for i in range(torch.cuda.device_count()):
   print(torch.cuda.get_device_properties(i).name)

In [ ]:
len( range(0, 1))

In [70]:
for i in range(0, 2):
    print(i)

0
1


In [72]:
import sys
print(sys.executable)

/usr/bin/python3.8


In [74]:
!bash /rsrch5/home/trans_mol_path/cercan/code/ACFormer/tools/dist_train.sh "/rsrch5/home/trans_mol_path/cercan/data/acformer/checkpoints/lizard/ACFormer_Lizard_finetune.py" 1

/opt/venv/jupyter/bin/python: Error while finding module specification for 'torch.distributed.launch' (ModuleNotFoundError: No module named 'torch')
